# Lab 16: Handling Missing Values and Outliers

Handling missing data and outliers is an important part of data preprocessing in data science and machine learning.

In this lab we will learn:

• How to detect missing values  
• How to perform mean and median imputation  
• How to detect outliers using statistical methods  
• How to treat outliers using IQR and Z-score  
• How to build a complete data cleaning pipeline

## Objectives

By the end of this lab, students will be able to:

- Understand the importance of data quality
- Detect missing values in datasets
- Apply mean and median imputation
- Identify outliers using statistical techniques
- Handle outliers using IQR and Z-score methods
- Implement a complete data cleaning pipeline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
# Set random seed
np.random.seed(42)

# Number of records
n_samples = 1000

# Generate base dataset
data = {
    'customer_id': range(1, n_samples + 1),
    'age': np.random.normal(35, 12, n_samples),
    'income': np.random.normal(50000, 15000, n_samples),
    'purchase_amount': np.random.normal(200, 50, n_samples),
    'satisfaction_score': np.random.normal(7, 1.5, n_samples),
    'years_customer': np.random.normal(3, 2, n_samples)
}

df = pd.DataFrame(data)

print("Dataset Created Successfully")
print(df.head())

In [ ]:
missing_indices_age = np.random.choice(df.index, size=50, replace=False)
missing_indices_income = np.random.choice(df.index, size=30, replace=False)
missing_indices_satisfaction = np.random.choice(df.index, size=40, replace=False)

df.loc[missing_indices_age, 'age'] = np.nan
df.loc[missing_indices_income, 'income'] = np.nan
df.loc[missing_indices_satisfaction, 'satisfaction_score'] = np.nan

In [ ]:
outlier_indices = np.random.choice(df.index, size=20, replace=False)

df.loc[outlier_indices, 'income'] = df.loc[outlier_indices, 'income'] * 3
df.loc[outlier_indices[:10], 'age'] = 90

In [ ]:
df['age'] = np.abs(df['age'])
df['income'] = np.abs(df['income'])
df['purchase_amount'] = np.abs(df['purchase_amount'])
df['years_customer'] = np.abs(df['years_customer'])

print("Dataset Shape:", df.shape)
df.head()

In [ ]:
print("Missing Values Count")

missing_counts = df.isnull().sum()

print(missing_counts)

In [ ]:
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_info = pd.DataFrame({
    'Missing_Count': missing_counts,
    'Missing_Percentage': missing_percentage
})

print(missing_info)

In [ ]:
plt.figure(figsize=(12,6))

plt.subplot(1,2,1)
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title("Missing Values Heatmap")

plt.subplot(1,2,2)
missing_counts.plot(kind='bar')
plt.title("Missing Values by Column")
plt.xticks(rotation=45)

plt.show()

In [ ]:
df_mean_imputed = df.copy()

numerical_cols = df.select_dtypes(include=[np.number]).columns

for col in numerical_cols:
    if df_mean_imputed[col].isnull().sum() > 0:
        mean_value = df_mean_imputed[col].mean()
        df_mean_imputed[col].fillna(mean_value, inplace=True)
        print(f"{col} filled with mean {mean_value}")

In [ ]:
df_median_imputed = df.copy()

for col in numerical_cols:
    if df_median_imputed[col].isnull().sum() > 0:
        median_value = df_median_imputed[col].median()
        df_median_imputed[col].fillna(median_value, inplace=True)
        print(f"{col} filled with median {median_value}")

In [ ]:
columns_to_plot = ['age','income','satisfaction_score']

fig, axes = plt.subplots(1,3, figsize=(15,5))

for i,col in enumerate(columns_to_plot):
    axes[i].hist(df_median_imputed[col], bins=30)
    axes[i].set_title(col)

plt.show()

In [ ]:
df_clean = df_median_imputed.copy()

print(df_clean.describe())

In [ ]:
numerical_columns = ['age','income','purchase_amount','satisfaction_score','years_customer']

plt.figure(figsize=(12,8))

for i,col in enumerate(numerical_columns):
    
    plt.subplot(2,3,i+1)
    sns.boxplot(y=df_clean[col])
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:
def detect_outliers_iqr(data, column):

    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = data[(data[column] < lower) | (data[column] > upper)]

    return outliers, lower, upper

In [ ]:
for col in numerical_columns:

    outliers, lower, upper = detect_outliers_iqr(df_clean, col)

    print(col)
    print("Outliers:", len(outliers))
    print("Lower Bound:", lower)
    print("Upper Bound:", upper)
    print()

In [ ]:
def detect_outliers_zscore(data, column, threshold=3):

    z_scores = np.abs(stats.zscore(data[column]))

    outliers = data[z_scores > threshold]

    return outliers

In [ ]:
for col in numerical_columns:

    outliers = detect_outliers_zscore(df_clean, col)

    print(col, "Outliers:", len(outliers))

In [ ]:
def cap_outliers_iqr(data, column):

    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    data[column] = data[column].clip(lower, upper)

    return data

In [ ]:
df_iqr_capped = df_clean.copy()

for col in ['age','income']:
    
    df_iqr_capped = cap_outliers_iqr(df_iqr_capped, col)

print("Outliers Capped Using IQR")

In [ ]:
plt.figure(figsize=(10,5))

plt.hist(df_iqr_capped['income'], bins=30)

plt.title("Income Distribution After Cleaning")

plt.show()

In [ ]:
df_iqr_capped.to_csv("cleaned_customer_data.csv", index=False)

print("Cleaned dataset exported successfully")

In [ ]:
print("Final Dataset Shape")

print(df_iqr_capped.shape)

print(df_iqr_capped.head())

## Conclusion

In this lab we explored techniques for handling missing values and outliers.

Key Learnings:

• Missing values can be detected using pandas methods  
• Mean and median imputation help fill missing values  
• Outliers can be detected using IQR and Z-score methods  
• Outliers can be treated using capping techniques  
• Data cleaning pipelines improve dataset quality

These techniques are essential in real-world data science workflows.